In [ ]:
import wget 
import pandas as pd
import xarray as xr
import numpy as np
import requests
from bs4 import BeautifulSoup
import wget
import os

import sys 


In [ ]:

# function to interpolate the wind and pressure timeseries in case of missing values (beginning and end of the track are removed in case of missing values)

def clean_and_interpolate_track(track,cols=['WMO_WIND kts', 'WMO_PRES mb']):
    track = track.sort_values('ISO_TIME')
    valid = track[cols].notna().any(axis=1)
    if not valid.any():
        return None
    first_idx = valid.idxmax()
    last_idx = valid[::-1].idxmax()
    track = track.loc[first_idx:last_idx].copy()
    track[cols] = track[cols].interpolate(method='linear')
    return track

#1.Download IBTrACS data from 1980 in csv format

In [ ]:


import os 
directory_ibtracs=''

url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.since1980.list.v04r01.csv'


file_name = 'ibtracs_since_1980.csv'
wget.download(url, os.path.join(directory_ibtracs,file_name))

2.Pre process IBTrACS data by splitting them according to the weather agency for responsible for them
2A. Clean name of the columns, remove 1st row (contains measure units for some column names)

In [ ]:
path_ibtracs=''
# file_name='ibTRACS_since_1980.csv'
df=pd.read_csv(os.path.join(path_ibtracs,file_name))
print('number of available cyclones',len(np.unique(df['SID'].values)))

#some processing to clean the first row of the dataset.
#it mostly contains measure units of some variables, that are included in the header,like 'WIND kts' or 'PRES mb'

first_row=[str(df.iloc[0].values[i]).strip() for i in range (len(df.iloc[0].values))]
df.columns = [
    (df.columns[i] + ' ' + first_row[i]).strip() if 'degrees' not in first_row[i]
    else df.columns[i].strip()
    for i in range(len(df.columns))
]
df=df.iloc[1:].reset_index(drop=True) #drop the first row after this processing


In [ ]:
valid_times=[str(n).zfill(2)+':00:00' for n in range (0,24,3)]
print('valid times',valid_times)
mask=df['ISO_TIME'].str[-8:].isin(valid_times)
df=df.loc[mask].reset_index(drop=True)

print('times in the dataset',np.unique(df['ISO_TIME'].str[-8:].values)) #check that data have all 3-h time resolution

print('n cyclones',len(np.unique(df['SID'].values)))
df['ISO_TIME2']=pd.to_datetime(df['ISO_TIME'])
df['time_gap'] = (df.sort_values(['SID', 'ISO_TIME2']).groupby('SID')['ISO_TIME2'].diff().dt.total_seconds() / 3600)
print('check that time gap between timesteps is either 3 (3 hours) or nan (in the case of the first timestep of each track):',np.unique(df['time_gap'].values))


In [ ]:
wind_cols=[]
lat_lon=[]
lat_lon=list(df.columns[np.logical_or(df.columns.str.contains('LAT'),df.columns.str.contains('LON'))])
wind_press=list(df.columns[np.logical_or(df.columns.str.contains('WIND'),df.columns.str.contains('PRES'))])
        
for col in wind_press+lat_lon:
    df[col]=pd.to_numeric(df[col],errors='coerce')

In [ ]:
agencies=['atcf', 'bom', 'hurdat_atl', 'hurdat_epa', 'nadi','newdelhi', 'reunion', 'tokyo', 'wellington']
agency_capitals=['USA','BOM','USA','USA','NADI','NEWDELHI','REUNION','TOKYO','WELLINGTON']
print(agencies,agency_capitals)
# Treat blank as missing agency name
df['WMO_AGENCY'] = (df['WMO_AGENCY'].astype(str).str.strip().replace('', np.nan))

# Unique non-NaN agencies per cyclone (SID is a unique cyclone identifier)
agency_per_sid = (df.groupby('SID')['WMO_AGENCY'].apply(lambda x: x.dropna().unique()))
# Keep only cyclones with exactly one agency
single_agency_sid = agency_per_sid[agency_per_sid.apply(len)==1]
#for each SID -> unique agency
sid_to_agency = single_agency_sid.apply(lambda x: x[0])
df_clean = df[df['SID'].isin(sid_to_agency.index)].copy()
print('n cyclones',len(np.unique(df_clean['SID'].values)))
df_clean['WMO_AGENCY'] = df_clean['WMO_AGENCY'].fillna(df_clean['SID'].map(sid_to_agency))
print('n cyclones',len(np.unique(df_clean['SID'].values)))
df_clean = df_clean.reset_index(drop=True)

#only basic columns that are needed for the analysis are kept
#Those include some general information on the track, the time and location info, and the wind and pressure data
cols_to_keep=['SID','SEASON Year','NUMBER','BASIN','SUBBASIN','NAME','ISO_TIME','NATURE','LAT','LON','time_gap'] 
for i in range (len(agencies)):
    print(agencies[i])
    df_save_agency=df_clean.loc[df_clean['WMO_AGENCY']==agencies[i]].reset_index(drop=True)
    col_keep_agency=cols_to_keep+list(df_clean.columns[df_clean.columns.str.contains(agency_capitals[i])])
    col_keep_agency=col_keep_agency+list(df_clean.columns[df_clean.columns.str.contains('WMO')])
    df_save_agency=df_save_agency[col_keep_agency]
    print(col_keep_agency)
    df_interp = (df_save_agency.groupby('SID', group_keys=True).apply(clean_and_interpolate_track, include_groups=False).reset_index(level=0))
    #df_interp=(df_save_agency.groupby('SID',group_keys=False).apply(clean_and_interpolate_track).reset_index(drop=True))
    print('len dataset',len(df_interp))
    #if i!=0:
        #leng=leng+len(np.unique(df_save['SID'].values))
    print('number cyclones',len(np.unique(df_interp['SID'].values)),'\n\n')   
    df_interp.to_csv(os.path.join(path_ibtracs,'dataset_ibtracs_basic_cols_'+agencies[i]+'.csv'))
    

In [ ]:
year=str(2005)
directory_ibtracs=''
ibtracs_filename='ibtracs_since_1980.csv' #csv containing the cyclone tracks
output_directory=os.path.join('',year)
os.makedirs(output_directory,exist_ok=True)
name_to_download='KATRINA'

df=pd.read_csv(os.path.join(directory_ibtracs+ibtracs_filename))
df=df.loc[df['NAME'].str.contains(name_to_download)].reset_index(drop=True)
times=df['ISO_TIME'].loc[df['ISO_TIME'].str.contains(year)].reset_index(drop=True)
times=times.str.replace('-','.')
times=times.str.split(':').str[0]
times=times.str.replace(' ','.')
times=np.unique(times)
#print(times)

url = "https://www.ncei.noaa.gov/data/geostationary-ir-channel-brightness-temperature-gridsat-b1/access/"+year+'/'

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
file_extensions = (".nc")
links = []

for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(file_extensions):
            if href.startswith("http"):
                links.append(href)
            else:
                links.append(url+href)
            

    #print(links)
for link in links:
        file=os.path.join(output_directory,link.split('/')[-1])
        #print(file)
        #print(file)
        with open(os.path.join(output_directory,'log_download.txt'), 'a') as f:
                    f.write(file)
                    f.write('\n')
                    f.close()
        if any(time in link for time in times) and os.path.exists(file)==False:
            #print(file)
            try:
                with open(os.path.join(output_directory,'log_download.txt'), 'a') as f:
                    f.write('downloading   '+link)
                    f.write('\n\n')
                    f.close()
   
                wget.download(link, out=output_directory)
            
            except:
                with open(os.path.join(output_directory,'log_ERRORS.txt'), 'a') as f:
                    f.write('failed downloading   '+link)
                    f.write('\n\n')
                    f.close()
                continue    
        else:
            with open(os.path.join(output_directory,'log_download.txt'), 'a') as f:
                    f.write('file already there '+file)
                    f.write('\n')
                    f.close()
            continue




#4.Crop IR image (224x224 points) of the given cyclone from GRIDSAT data

In [ ]:
agencies=['hurdat_atl', 'hurdat_epa', 'nadi',
       'newdelhi', 'reunion', 'tokyo', 'wellington']
agencies_capital=['USA','BOM','USA','USA','NADI','NEWDELHI','REUNION','TOKYO','WELLINGTON']

agency_index=0   #hurdat_atl is selected to check KATRINA hurricane as an exampke
agency=agencies[agency_index]
year=str(2005)
path_output=''
path_gridsat_data=''
path_ibtracs=''
path_output=''
#open dataset and ensure variables are numeric
df_tot=pd.read_csv(os.path.join(path_ibtracs,'dataset_ibtracs_basic_cols_'+agency+'.csv'))
var_press=agencies_capital[agency_index]+'_PRES mb'

df_tot[var_press]=pd.to_numeric(df_tot[var_press],errors='coerce')
var_wind=agencies_capital[agency_index]+'_WIND kts'

df_tot[var_wind]=pd.to_numeric(df_tot[var_wind],errors='coerce')


# In[11]:


#IBTRACS use both the -180-180 and the 0-360 longitude formats depending on the agency
#here all the longitude values are transformed into 

df_tot['LON']=(df_tot['LON'].values + 180) % 360 - 180
if os.path.exists(os.path.join(path_output,'log_ERRORS.txt'))==True:
    os.remove(os.path.join(path_output,'log_ERRORS.txt'))

#for this demo, again we keep only 2005
start_year=2005
end_year=2006

for year in [str(t) for t in range (start_year,end_year)]:
    
    df_year=df_tot.loc[df_tot['ISO_TIME'].str.contains(year)].reset_index(drop=True)
    if len(df_year)==0:
        continue

    #UNNAMED cyclones must be renamed adding a numeric label: UNNAMED_0, UNNAMED_1 etc
    UNNAMED=df_year[df_year['NAME']=='UNNAMED'].reset_index(drop=True)
    df_year=df_year.loc[df_year['NAME']!='UNNAMED'].reset_index(drop=True)
    SIDs_UNNAMED=np.unique(UNNAMED["SID"].values)
    p=-1
    for SID in SIDs_UNNAMED:
        p+=1
        UNNAMED.loc[UNNAMED["SID"]==SID,'NAME']=UNNAMED['NAME']+str(p)
    df_year=pd.concat([df_year,UNNAMED],axis=0).reset_index(drop=True)
    names=np.unique(df_year['NAME'].values)
    
    
    half_width=7.5
    half_width_points=112
    #JUST KEEP KATRINA AS AN EXAMPLE
    names=[names[i] for i in range(len(names)) if 'KATRINA' in names[i]]
    
    for name in names:
                
            df=df_year.loc[df_year['NAME']==name].reset_index(drop=True)
            times2=df['ISO_TIME']
            times=df['ISO_TIME'].str.split(':').str[0]
            times=times.str.replace(' ','.')
            times=times.str.replace('-','.')
        
            j=-1
            cyclone_full=[]
            file_missings=[]
            for time1 in times:
                j+=1
                file=path_gridsat_data+year+'/GRIDSAT-B1.'+time1+'.v02r01.nc'
                if os.path.exists(file)==False:
                    file_missings.append(file.split('B1.')[1].split('.v02r01.nc')[0])

                try:
                    
                    with open(os.path.join(path_output,'log_download.txt'), 'a') as f:
                            f.write('opening '+file)
                            f.write('\n')
                            f.close()
                    ds=xr.open_dataset(file)
                    ds=ds[['irwin_cdr']]
                    lat_ds=ds.lat.values
                    lon_ds=ds.lon.values
                    grid_spacing_lat=np.nanmean(np.diff(lat_ds))
                    grid_spacing_lon=np.nanmean(np.diff(lon_ds))
            
                    lat_cen=df['LAT'][df['ISO_TIME']==times2.iloc[j]].values[0]
                    lon_cen=df['LON'][df['ISO_TIME']==times2.iloc[j]].values[0]
                    press=df[var_press][df['ISO_TIME']==times2.iloc[j]].values[0]
                    wind=df[var_wind][df['ISO_TIME']==times2.iloc[j]].values[0]
                    lat_cen_index=np.argmin(np.abs(lat_ds-lat_cen))
                    lon_cen_index=np.argmin(np.abs(lon_ds-lon_cen))
                    nx=len(lon_ds)

                    #the selection of the points in the longitudinal direction must be periodic across the boundary
                    lon_sel=(np.arange(lon_cen_index - half_width_points, lon_cen_index+half_width_points) % nx)
                    lat_sel=(np.arange(lat_cen_index - half_width_points, lat_cen_index+half_width_points))

                    
                    
                    cycl=ds.isel(lon=xr.DataArray(lon_sel, dims="lon"),lat=xr.DataArray(lat_sel, dims="lat"))
                    
                    #lat and lon are redefined as difference from the coordinates of the center
                    new_lat=np.arange(-half_width_points*grid_spacing_lat,half_width_points*grid_spacing_lat,grid_spacing_lat)
                    new_lon=np.arange(-half_width_points*grid_spacing_lon,half_width_points*grid_spacing_lon,grid_spacing_lon)

                    cycl = cycl.assign_coords(lat=new_lat,lon=new_lon)
                    
                    cycl['LON center'] = xr.DataArray([lon_cen],dims="time",coords={"time":cycl.time.values})
                    cycl['LAT center'] = xr.DataArray([lat_cen],dims="time",coords={"time":cycl.time.values})
                    cycl['Min pressure mb'] = xr.DataArray([press],dims="time",coords={"time":cycl.time.values})
                    cycl['Max wind kts'] = xr.DataArray([wind],dims="time",coords={"time":cycl.time.values})
                    
                    
                  
                    #shape of (lat,lon) is supposed to be (224,224) for every snapshot
                    if np.shape(cycl['irwin_cdr'])!=(1,224,224):
                        
                        with open(os.path.join(path_output,'/log_ERRORS.txt'), 'a') as f:
                            f.write('different shape '+name+'   '+year+' '+time1)
                            f.write('  ')
                            f.write('shape '+str(np.shape(cycl)))
                            f.write('  ')
                            f.write('lat cen  '+str(lat_cen)+' lon cen '+str(lon_cen))
                            f.write('\n')
                            
                    
                    cyclone_full.append(cycl)
                    
                except Exception as e:
                    with open('log_ERRORS.txt', 'a') as f:
                        f.write(year+'_'+name+time1)
                        f.write('\n')
                        if hasattr(e, "filename") and e.filename is not None:
                            f.write(f"{type(e).__name__}: {e.filename}\n")
                        else:
                            f.write(f"{type(e).__name__}: {e}\n")
                      
                    continue

            cyclone_full=xr.concat(cyclone_full[:],dim='time')
            cyclone_full.attrs['files_missing']=file_missings
            cyclone_full.to_netcdf(os.path.join(path_output,year+'_'+name+'.nc'))

